In [5]:
# Install & clone

import os

!pip install hnswlib transformers accelerate -q

REPO_DIR = "/kaggle/working/vision-search-engine"
if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git pull
else:
    !git clone https://github.com/3-pi-radians/vision-search-engine.git {REPO_DIR}
    %cd {REPO_DIR}

!git log --oneline -3

/kaggle/working/vision-search-engine
remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 3 (delta 2), reused 3 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 848 bytes | 848.00 KiB/s, done.
From https://github.com/3-pi-radians/vision-search-engine
   1af9a83..9b9db6d  main       -> origin/main
Updating 1af9a83..9b9db6d
Fast-forward
 config.py | 64 ++++++++++++++++++++++++++++++++++++++-------------------------
 1 file changed, 39 insertions(+), 25 deletions(-)
9b9db6d (HEAD -> main, origin/main, origin/HEAD) Updated config.py for HNSW index
1af9a83 Added file to build HNSW index
4154305 Fix CLIP embedding extraction: use vision_model/text_model + projection explicitly


In [6]:
# Verify all inputs

import os

checks = {
    "image_paths.json":  "/kaggle/input/datasets/pankajdeopa/deepfashion-inshop-crops/image_paths.json",
    "captions.json":     "/kaggle/input/datasets/pankajdeopa/deepfashion-inshop-captions/captions.json",
    "clip_finetuned.pt": "/kaggle/input/datasets/pankajdeopa/deepfashion-clip-weights/clip_finetuned.pt",
    "crops/gallery":     "/kaggle/input/datasets/pankajdeopa/deepfashion-inshop-crops/crops/gallery",
}

all_ok = True
for name, path in checks.items():
    exists = os.path.exists(path)
    size = f"({os.path.getsize(path)/1e6:.1f} MB)" if exists and os.path.isfile(path) else ""
    print(f"  {name}: {'✅' if exists else '❌ MISSING'} {size}")
    if not exists:
        all_ok = False

print("\n✅ All inputs ready — run Cell 3" if all_ok else "\n❌ Attach missing datasets before running")

  image_paths.json: ✅ (1.5 MB)
  captions.json: ✅ (0.3 MB)
  clip_finetuned.pt: ✅ (605.3 MB)
  crops/gallery: ✅ 

✅ All inputs ready — run Cell 3


In [7]:
#  Verify config paths

!grep -n "IMAGE_PATHS_PATH\|CAPTIONS_PATH\|CLIP_WEIGHTS_DIR\|CROPS_DIR" \
    /kaggle/working/vision-search-engine/config.py

15:    CROPS_DIR          = Path("/kaggle/input/datasets/pankajdeopa/deepfashion-inshop-crops/crops")
16:    IMAGE_PATHS_PATH   = Path("/kaggle/input/datasets/pankajdeopa/deepfashion-inshop-crops/image_paths.json")
17:    CAPTIONS_PATH      = Path("/kaggle/input/datasets/pankajdeopa/deepfashion-inshop-captions/captions.json")
18:    CLIP_WEIGHTS_DIR   = Path("/kaggle/input/datasets/pankajdeopa/deepfashion-clip-weights")
22:    CROPS_DIR          = Path("data/working/crops")
23:    IMAGE_PATHS_PATH   = Path("data/working/image_paths.json")
24:    CAPTIONS_PATH      = Path("data/working/captions.json")
25:    CLIP_WEIGHTS_DIR   = Path("data/working/clip_weights")


In [8]:
# Run HNSW index build

!python offline/run_build_index.py

2026-04-28 14:28:29,870 INFO Gallery size: 12612
2026-04-28 14:28:29,879 INFO Loaded 3985 captions
2026-04-28 14:28:29,879 INFO Device: cuda
2026-04-28 14:28:30,036 INFO HTTP Request: GET https://huggingface.co/api/models/openai/clip-vit-base-patch32/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-04-28 14:28:30,096 INFO HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-04-28 14:28:30,158 INFO HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/chat_template.json "HTTP/1.1 404 Not Found"
2026-04-28 14:28:30,218 INFO HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"
2026-04-28 14:28:30,281 INFO HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/audio_tokenizer_config.json "HTTP/1.1 404 Not Found"
2026-04-28 14:28:3

In [9]:
# Publish as kaggle dataset

import json, os, shutil

USERNAME = "pankajdeopa"
PUBLISH_DIR = "/kaggle/working/hnsw-dataset"
os.makedirs(PUBLISH_DIR, exist_ok=True)

# Copy indexes
for cfg in ["A", "B", "C"]:
    src = f"/kaggle/working/hnsw_index_{cfg}.bin"
    dst = f"{PUBLISH_DIR}/hnsw_index_{cfg}.bin"
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f" Copied hnsw_index_{cfg}.bin")
    else:
        print(f" hnsw_index_{cfg}.bin not found — run Cell 4 first")

# Copy image_paths.json (needed by runtime for item_id lookup)
shutil.copy(
    "/kaggle/input/datasets/pankajdeopa/deepfashion-inshop-crops/image_paths.json",
    f"{PUBLISH_DIR}/image_paths.json"
)
print("  Copied image_paths.json")

# Write metadata
meta = {
    "title": "deepfashion-hnsw-indexes",
    "id": f"{USERNAME}/deepfashion-hnsw-indexes",
    "licenses": [{"name": "CC0-1.0"}]
}
with open(f"{PUBLISH_DIR}/dataset-metadata.json", "w") as f:
    json.dump(meta, f, indent=2)

print("\nPublishing...")
!kaggle datasets create -p {PUBLISH_DIR} --dir-mode zip

print(f"\n Published: https://www.kaggle.com/datasets/{USERNAME}/deepfashion-hnsw-indexes")

  ✅ Copied hnsw_index_A.bin
  ✅ Copied hnsw_index_B.bin
  ✅ Copied hnsw_index_C.bin
  Copied image_paths.json

Publishing...
Starting upload for file hnsw_index_A.bin
100%|██████████████████████████████████████| 26.4M/26.4M [00:00<00:00, 43.7MB/s]
Upload successful: hnsw_index_A.bin (26MB)
Starting upload for file hnsw_index_C.bin
100%|██████████████████████████████████████| 26.4M/26.4M [00:00<00:00, 39.4MB/s]
Upload successful: hnsw_index_C.bin (26MB)
Starting upload for file hnsw_index_B.bin
100%|██████████████████████████████████████| 26.4M/26.4M [00:00<00:00, 36.0MB/s]
Upload successful: hnsw_index_B.bin (26MB)
Starting upload for file image_paths.json
100%|██████████████████████████████████████| 1.45M/1.45M [00:00<00:00, 4.00MB/s]
Upload successful: image_paths.json (1MB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/pankajdeopa/deepfashion-hnsw-indexes

 Published: https://www.kaggle.com/datasets/pankajdeopa/deepfashion-hnsw-index

In [10]:
# Summary


import os

print("=== SCRUM-23 Complete — All offline artifacts ===\n")

datasets = {
    "deepfashion-inshop-crops":      "/kaggle/input/datasets/pankajdeopa/deepfashion-inshop-crops/image_paths.json",
    "deepfashion-inshop-captions":   "/kaggle/input/datasets/pankajdeopa/deepfashion-inshop-captions/captions.json",
    "deepfashion-clip-weights":      "/kaggle/input/datasets/pankajdeopa/deepfashion-clip-weights/clip_finetuned.pt",
    "deepfashion-hnsw-indexes (A)":  "/kaggle/working/hnsw-dataset/hnsw_index_A.bin",
    "deepfashion-hnsw-indexes (B)":  "/kaggle/working/hnsw-dataset/hnsw_index_B.bin",
    "deepfashion-hnsw-indexes (C)":  "/kaggle/working/hnsw-dataset/hnsw_index_C.bin",
}

for name, path in datasets.items():
    exists = os.path.exists(path)
    size = f"({os.path.getsize(path)/1e6:.1f} MB)" if exists else ""
    print(f"  {'✅' if exists else '❌'} {name} {size}")

print("\n Offline pipeline complete")

=== SCRUM-23 Complete — All offline artifacts ===

  ✅ deepfashion-inshop-crops (1.5 MB)
  ✅ deepfashion-inshop-captions (0.3 MB)
  ✅ deepfashion-clip-weights (605.3 MB)
  ✅ deepfashion-hnsw-indexes (A) (27.7 MB)
  ✅ deepfashion-hnsw-indexes (B) (27.7 MB)
  ✅ deepfashion-hnsw-indexes (C) (27.7 MB)

 Offline pipeline complete
